<a href="https://colab.research.google.com/github/jiminmini/mini/blob/ESAA_OB/9_22_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B0_%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**[주제 및 데이터]**
**주제**:

건물의 전력·가스·증기·냉수 사용량(에너지 소비)을 예측하는 문제.
특히 대형 빌딩 단위에서 “다음 기간의 에너지 사용량”을 정확히 추정하는 것이 목표였음

**데이터 설명**:
- 학습 데이터: 건물별 시간 단위 에너지 사용량 (meter reading)

- 피처:

건물 ID, 계량기 타입(전기, 가스, 증기, 냉수)

기상 데이터(온도, 습도, 풍속 등)

건물 메타데이터(건물 크기, 1차 사용 목적, 건물 연도 등)

시간 변수(년, 월, 요일, 시간 등)

- 특징: 결측치 많음, 다양한 단위와 범위, 건물·지역별 특성이 크게 달라서 전처리가 핵심
---


#**[코드 리뷰]**
**1. EDA (탐색적 데이터 분석)**
- 계량기 타입별 데이터 분포 파악: 전기 사용량이 가장 많고, 가스·증기·냉수는 건물별 편차가 큼

- 기상 데이터와의 상관관계 확인: 온도/습도와 에너지 소비량이 계절별로 뚜렷한 상관관계 존재

- 건물별 사용량 패턴 시각화: 요일·시간별 주기성, 휴일 효과 등을 확인

---


**2. 전처리**
- 결측치 처리:

날씨 데이터 보간(Interpolation)

건물 메타데이터 결측치 채움 (평균값, 모드 대체)

- 피처 엔지니어링:

날짜 파생 변수 생성(월, 요일, 시간, 계절)

날씨의 이동평균/지연(lag) 값 생성 (예: 이전 3시간 평균 온도)

건물별 normalize / 로그 변환 (target 안정화)

- 스케일링: 일부 변수는 log1p 변환으로 안정성 확보

- 데이터 병합: 건물 메타데이터 + 기상데이터 + 에너지 사용량을 ID·시간 기준으로 조인

---
**3. 모델링 (Modeling)**

- 주요 알고리즘:

LightGBM & XGBoost (Gradient Boosting Machines) → 가장 우수한 성능

CatBoost도 일부 팀에서 활용했으나 LightGBM이 속도·성능 면에서 우세

- 모델 학습 전략:

meter type별/region별로 모델 분할 학습 → 데이터 이질성 해결

KFold validation (시간 기반 분할)

Feature importance 분석으로 불필요한 변수 제거

- 앙상블:

다수의 LightGBM 모델 결과를 평균(blending) → 일반화 성능 개선

상위팀은 stacking 대신 단순 blending이 더 안정적이라고 보고함

---
#**[차별 점 및 배울 점]**

✅ 차별 점
- 단순히 모델링보다 정교한 전처리 + 피처 엔지니어링이 성능을 좌우

- 건물·지역 특성을 고려해 데이터를 세분화해서 모델링한 점이 주효

- Gradient Boosting 모델을 최적화하면서도 계산 효율성을 높이기 위해 LightGBM 중심으로 사용

✨ 배울 점
- 복잡한 모델보다 데이터 전처리와 도메인 지식 기반 피처 생성이 훨씬 중요

- 시계열 데이터를 다룰 때는 단순한 lag/rolling feature가 큰 힘을 발휘

- 다양한 모델을 무리하게 섞기보다 간단한 앙상블 전략이 오히려 좋은 결과를 줄 수 있음

- 캐글 수상작 리뷰를 통해 “모델 성능 = 데이터 이해 + 전처리 + 피처 엔지니어링”이라는 사실을 다시 확인 가능
